# Task 4 – SPAM/HAM Classification Model

## Methodology

### Text Representation – TF-IDF with Bigrams

$$\text{TF-IDF}(w, d) = \text{TF}(w, d) \times \log\frac{N}{df(w)}$$

- **TF**: word frequency in this message (log-scaled: `sublinear_tf=True`)
- **IDF**: rarity across all messages — down-weights words that appear everywhere
- **Bigrams** included (`ngram_range=(1,2)`) to capture phrases like *"free entry"*

### Models

| Model | How it works |
|-------|--------------|
| **Logistic Regression** (primary) | Learns a weight per word; interpretable coefficients |
| **Naive Bayes** (baseline) | Probabilistic: `P(SPAM\|words)`; classic spam filter approach |

### Why not accuracy?
The dataset is **imbalanced** (86 % HAM / 14 % SPAM). A model that always predicts HAM
achieves 86 % accuracy without learning anything. Primary metrics: **F1-SPAM** and **ROC-AUC**.

### Pipeline
Preprocessing is baked into the sklearn Pipeline — identical transformations
at training and inference time, preventing train-serve skew.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

from sms_analysis import load_dataset
from sms_analysis.classification import SpamClassifier, evaluate, cross_validate
from sms_analysis.classification.pipeline import _build_pipeline
from sms_analysis.visualization import plot_confusion_matrix, plot_roc_curve, save_figure

DATASET = Path("../assignment/3D8026C4.zip/SMSSpamCollection")
OUTPUTS = Path("../outputs")

dataset = load_dataset(DATASET)

X_train, X_test, y_train, y_test = train_test_split(
    dataset.texts, dataset.labels,
    test_size=0.2, stratify=dataset.labels, random_state=42,
)
print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")
print(f"SPAM in test: {y_test.count('spam')} / {len(y_test)} ({y_test.count('spam')/len(y_test):.1%})")

Train: 4457  |  Test: 1115
SPAM in test: 149 / 1115 (13.4%)


## Train both models

In [2]:
clf_lr = SpamClassifier("logistic_regression").fit(X_train, y_train)
clf_nb = SpamClassifier("naive_bayes").fit(X_train, y_train)

res_lr = evaluate(y_test, clf_lr.predict(X_test), clf_lr.predict_proba(X_test))
res_nb = evaluate(y_test, clf_nb.predict(X_test), clf_nb.predict_proba(X_test))

comparison = pd.DataFrame([
    {"Model": "Logistic Regression", "Accuracy": res_lr.accuracy,
     "F1-SPAM": res_lr.f1_spam, "Precision": res_lr.precision_spam,
     "Recall": res_lr.recall_spam, "ROC-AUC": res_lr.roc_auc},
    {"Model": "Naive Bayes", "Accuracy": res_nb.accuracy,
     "F1-SPAM": res_nb.f1_spam, "Precision": res_nb.precision_spam,
     "Recall": res_nb.recall_spam, "ROC-AUC": res_nb.roc_auc},
]).set_index("Model").round(3)
comparison

,Accuracy,F1-SPAM,Precision,Recall,ROC-AUC
Model,,,,,
Logistic Regression,0.977,0.906,0.984,0.839,0.993
Naive Bayes,0.970,0.875,1.000,0.779,0.989


## Cross-validation (Logistic Regression)

In [3]:
cv_result = cross_validate(_build_pipeline("logistic_regression"), dataset.texts, dataset.labels)
print(f"5-fold CV  F1-macro: {cv_result['f1_mean']:.3f} ± {cv_result['f1_std']:.3f}")

5-fold CV  F1-macro: 0.928 ± 0.010


## Confusion Matrix

In [4]:
fig = plot_confusion_matrix(res_lr.confusion)
save_figure(fig, OUTPUTS / "04_confusion_matrix.png")

## ROC Curve

In [5]:
fig = plot_roc_curve(y_test, clf_lr.predict_proba(X_test))
save_figure(fig, OUTPUTS / "04_roc_curve.png")

## Feature Importance – What drives SPAM vs HAM?

In [6]:
import matplotlib.pyplot as plt
import seaborn as sns

features = clf_lr.top_features(n=15)

import pandas as pd
df_spam_f = pd.DataFrame(features["spam"], columns=["word", "coefficient"])
df_ham_f  = pd.DataFrame(features["ham"],  columns=["word", "coefficient"])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.barplot(data=df_spam_f, x="coefficient", y="word", ax=axes[0], color="tomato")
sns.barplot(data=df_ham_f,  x="coefficient", y="word", ax=axes[1], color="steelblue")
axes[0].set_title("Top SPAM features (LR coefficients)", fontweight="bold")
axes[1].set_title("Top HAM features (LR coefficients)",  fontweight="bold")
for ax in axes:
    ax.set_xlabel("Coefficient")
    ax.set_ylabel("")
plt.tight_layout()
save_figure(fig, OUTPUTS / "04_feature_importance.png")

## Live prediction demo

In [7]:
examples = [
    "WINNER! You have been selected for a FREE cash prize. Call 0800 now!",
    "Hey, are you coming to the cinema tonight?",
    "Claim your £500 reward at www.win-now.com immediately!",
    "I'll be home around 7, can you start dinner?",
]
predictions = clf_lr.predict(examples)
probabilities = clf_lr.predict_proba(examples)

for text, pred, proba in zip(examples, predictions, probabilities):
    print(f"[{pred.upper():4}] p(spam)={proba[1]:.2%}  |  {text[:60]}")

[SPAM] p(spam)=86.32%  |  WINNER! You have been selected for a FREE cash prize. Call 0
[HAM ] p(spam)=2.94%  |  Hey, are you coming to the cinema tonight?
[SPAM] p(spam)=81.54%  |  Claim your £500 reward at www.win-now.com immediately!
[HAM ] p(spam)=2.14%  |  I'll be home around 7, can you start dinner?


## Summary

| Metric | Logistic Regression | Naive Bayes |
|--------|--------------------|--------------|
| F1-SPAM | ~0.91 | ~0.88 |
| ROC-AUC | ~0.99 | ~0.98 |

Both models perform excellently. LR is preferred because:
1. **Interpretable coefficients** — directly shows which words drive SPAM/HAM decisions
2. **Slightly higher F1 and AUC** on this dataset
3. **Calibrated probabilities** — useful for threshold adjustment in production

The preprocessing pipeline ensures that a real-world SMS containing a URL
(`http://win-now.com`) is handled identically to a training example with `&lt;URL&gt;`.